# Build a Knowledge Graph for Your MPLS L3VPN — the guided version

*A warm, step-by-step lab for network engineers. You bring the MPLS. We bring the graph.*

MPLS turns your core into a set of **label-switched paths** — tunnels that carry customer
VPNs across provider routers that hold **no customer state at all**. That is the magic of
the technology, and also its quietest danger: the router that fails may be one you never
configured for that customer. A **knowledge graph** makes that hidden dependency explicit —
so a core box with zero customer config can still, on the graph, *name the customer it takes
down*.

By the end of this notebook you will have built, from an empty page, a small graph that
answers the question no `show mpls forwarding-table` can:

> *"p-core-2 just lost its labels. Which paying customer is now at risk?"*

and watched it print **`Acme Payments`** — a fact that lives in nobody's forwarding table,
least of all on the core router that broke.

### First, calm the nerves: this is **not** machine learning
No training. No model. No embeddings. No AI guessing in the dark. A knowledge graph is just
**structured facts** (things, and the named links between them) plus **queries** that walk
those links. Everything is **deterministic and inspectable** — run it twice, get the same
answer, and you can point at every node that produced it. If you can read an MPLS forwarding
table, you can read every line in this lab.

### What you need
Nothing. This runs in **Google Colab** with zero setup, using plain **Python + networkx +
matplotlib** (both already installed in Colab). No database, no Docker, no credentials.
Press *Runtime -> Run all*, or run each cell in order with **Shift+Enter**.

> **Coming from the OSPF lab?** Same five moves, one layer deeper. There, a service depended
> on a *prefix in an area*. Here, a service depends on a *tunnel across the core* — and that
> tunnel, as you'll see at the very end, is itself built over the OSPF underlay you already
> modeled.


## The whole vocabulary — the MPLS things you already know, as graph words

Before any code, here is the *entire* mental model. Everything after this is these ideas,
repeated. And notice: you don't have to learn what any of them *mean* — you only have to
learn which MPLS thing each one maps onto.

| Graph word | Plain meaning | The MPLS thing it already is |
|---|---|---|
| **node** | a thing | a PE, a P router, an LSP, a VRF, a customer service |
| **edge** | a named, directed link between two nodes | "this VRF rides that LSP", "this LSP transits that P" |
| **label** | a node's *type* | `PE`, `P`, `LSP`, `VRF`, `Service` — the role a box or object plays |
| **property** | a fact stored *on* a node or edge | `state='down'`, `criticality='critical'` |
| **traversal** | walking edges to answer a question | following the tunnel from the broken hop to the customer |

Two of those are worth a sentence each, because MPLS lives in the space between them:

- **overlay** — the customer's world: the **Service**, its **VRF**, the **PE** it terminates on.
  This is where all the customer config lives.
- **underlay** — the provider's transport: the **LSP** and the **P** routers it is
  label-switched across. This holds *no customer config* — just labels.

One idea deserves a spotlight, because it is the whole trick: **a fact can live on an edge,
not just a node.** "This hop lost its labels" is not really a property of the LSP or of the
P router — it is a property of *the LSP crossing that particular hop*. So we write it down
literally: the failure lives **on the `TRANSITS` edge**. Keep an eye out for that moment — it
is where a pretty tunnel diagram turns into something you can *query*.


## Setup — one import, one empty graph

**Theory (10 seconds).** `networkx` is a tiny Python library for building graphs in memory.
We will use a **`DiGraph`** — a *directed* graph, where every edge has a direction, an arrow.
That matters to us: `vrf-acme -> lsp-blue` ("this VRF rides that LSP") is a different
statement from `lsp-blue -> vrf-acme`. MPLS is full of directional truth — who imposes a
label, which way a path is signalled — so directed edges are the honest choice.

Run the next cell. It imports the tools and hands you an empty graph to fill.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Both libraries ship pre-installed in Colab — nothing to pip install.
# A DiGraph is a *directed* graph: every edge is an arrow, from one node to another.
G = nx.DiGraph()

print('Empty graph ready:', G)
print('Nodes:', G.number_of_nodes(), ' Edges:', G.number_of_edges())

**Checkpoint.** You should see `Empty graph ready` and `Nodes: 0  Edges: 0`. That empty
`G` is your blank whiteboard. Every step from here just adds nodes and edges to it.


## Step 1 — The provider fabric: PE and P routers, as nodes

**Theory.** In an MPLS provider network, routers play two very different jobs, and the whole
lesson of this lab hangs on the difference:

- a **PE** (provider *edge*) is where the customer plugs in. It holds the customer's **VRF**
  — their private routing table — imposes VPN labels, and knows the customer by name.
- a **P** (provider *core*) router only *swaps labels*. It forwards packets deeper into the
  core by label alone and holds **no customer VRF, no customer route, no customer anything**.
  It has never heard of your customer.

That asymmetry is the entire point. When a P router in the middle of the core has a bad day,
nothing on *it* mentions a customer — yet a customer goes dark. Keep that tension in mind; the
graph is what will resolve it.

In graph terms, each router becomes a **node** whose **label** records its role: `PE` or `P`.
We model two edge routers and two core routers — the smallest fabric that still has a *core to
cross*.


In [ ]:
# add_node(name, **properties). The name is a unique handle; label records the role.
G.add_node('pe-west',  label='PE')   # customer edge — holds VRFs
G.add_node('pe-east',  label='PE')   # customer edge — holds VRFs
G.add_node('p-core-1', label='P')    # pure core — swaps labels, no customer state
G.add_node('p-core-2', label='P')    # pure core — swaps labels, no customer state

print(G.number_of_nodes(), 'nodes so far:', list(G.nodes))
for n, d in G.nodes(data=True):
    print(f'  {n}: label={d["label"]}')

**Checkpoint.** Four nodes: two `PE` and two `P`. They're floating unconnected for now — bare
actors on the stage. Notice we stored *nothing* about customers on the `P` routers, because in
real life there is nothing to store. The next steps add the customer's world — and it will all
hang off the edges (PE), never the core (P).


## Step 2 — The overlay: the customer's world, terminated at the edge

**Theory.** Now the customer. An L3VPN customer shows up in three linked pieces, all living at
the *edge*:

- a **Service** — the business thing that must keep working. Here, **Acme Payments**. We tag it
  `criticality='critical'`, because that is exactly the fact no routing protocol carries.
- a **VRF** — the customer's private routing table on a PE (`vrf-acme`). Their isolated slice of
  the L3VPN.
- and the **PE** it sits on (`pe-east`, from Step 1).

We wire them with two edges that read like plain English:

- `Acme Payments -RUNS_IN-> vrf-acme` — *"this service runs inside that VRF."*
- `vrf-acme -ON-> pe-east` — *"that VRF lives on this PE."*

This is the overlay — everything the customer touches, all of it anchored to a single PE. Not
one byte of it lives in the core. That is deliberate, and it's about to become the whole story.


In [ ]:
G.add_node('Acme Payments', label='Service', criticality='critical')
G.add_node('vrf-acme',      label='VRF')

G.add_edge('Acme Payments', 'vrf-acme', rel='RUNS_IN')
G.add_edge('vrf-acme',      'pe-east',  rel='ON')

print('Overlay edges so far:')
for u, v, d in G.edges(data=True):
    print(f'  {u} --{d["rel"]}--> {v}')

**Checkpoint.** Two edges print: `Acme Payments -RUNS_IN-> vrf-acme` and `vrf-acme -ON-> pe-east`.
The customer now exists in the graph and is pinned to `pe-east`. But a VRF on a PE can't reach
the *other* side of the VPN on its own — it needs a way across the core. That way is the node
MPLS makes essential, and it's next.


## Step 3 — The LSP: the transport tunnel MPLS makes essential

**Theory.** Here is the node that has no equivalent in a plain IP diagram, and the reason this
lab is a *layer deeper* than the OSPF one. A VPN does not travel over "the network" in the
abstract — it travels inside a specific **label-switched path (LSP)**: a tunnel, built across
the core, that the PE pushes the customer's traffic into.

So we add an **LSP** node, `lsp-blue`, with a `state` property (`up` for now), and we connect
it to the overlay with two edges:

- `vrf-acme -USES_LSP-> lsp-blue` — *"this VRF's traffic rides that tunnel."* This is the
  single strand that ties the customer to the transport. Cut it and the customer is stranded,
  no matter how healthy their PE is.
- `pe-east -HAS_LSP-> lsp-blue` — *"this PE is the head-end of that tunnel."*

Notice the discipline: **one node for the whole tunnel.** We are not modelling every label
value, every LFIB entry, every swap. We store the *shape* of the dependency — that a customer
rides this path — and leave the forwarding-table firehose in the routers where it belongs.
We'll come back to that principle by name later.


In [ ]:
G.add_node('lsp-blue', label='LSP', state='up')

G.add_edge('vrf-acme', 'lsp-blue', rel='USES_LSP')
G.add_edge('pe-east',  'lsp-blue', rel='HAS_LSP')

print('The load-bearing strand:',
      [f'{u} -USES_LSP-> {v}' for u, v, d in G.edges(data=True) if d['rel'] == 'USES_LSP'])
print('LSP facts:', G.nodes['lsp-blue'])

**Checkpoint.** The `lsp-blue` node exists with `state='up'`, and `vrf-acme -USES_LSP-> lsp-blue`
now ties the customer to a transport tunnel. But the tunnel is still just a name — it doesn't yet
*cross* anything. The next step is where MPLS gets real, and where the failure will live.


## Step 4 — The underlay: the LSP transits the core, hop by hop (the pivotal step)

**Theory.** Here is the idea the whole lab pivots on. An LSP isn't a magic wire; it is
label-switched **hop by hop** across P routers. Each hop either has the labels it needs, or it
doesn't. So *where* does "this hop is broken" belong?

Not on the LSP as a whole (most of it is fine). Not on the P router as a whole (it's forwarding
plenty of other labels). It belongs on the **relationship between them** — *this LSP, crossing
that specific hop*. MPLS already agrees with you: a label binding is per-LSP, per-hop.

So we add **edges** with a `rel` of `TRANSITS`, and we hang a `state` **property** right on each
one. Read `lsp-blue -TRANSITS(down)-> p-core-2` literally: *"the blue LSP crosses p-core-2, and
at that hop it is down."* The wiring:

- `lsp-blue -TRANSITS-> p-core-1` is **up** — labels present, this hop forwards fine.
- `lsp-blue -TRANSITS-> p-core-2` is **down** — this hop lost its labels. **This one edge is the
  incident.**

And the rule that makes it bite: **one dead hop breaks the whole path.** A tunnel is only as
good as its worst hop. `p-core-2` has no customer config on it whatsoever — yet by breaking one
`TRANSITS` edge, it is about to take a named customer offline.


In [ ]:
# add_edge(source, target, **properties). The failure is a property ON the edge.
G.add_edge('lsp-blue', 'p-core-1', rel='TRANSITS', state='up')
G.add_edge('lsp-blue', 'p-core-2', rel='TRANSITS', state='down')   # <-- the whole incident

for u, v, d in G.edges(data=True):
    if d.get('rel') == 'TRANSITS':
        print(f'{u} --TRANSITS({d["state"]})--> {v}')
print('Graph now has', G.number_of_nodes(), 'nodes and', G.number_of_edges(), 'edges.')

**Checkpoint.** Two `TRANSITS` edges print, and exactly one reads `down`: the
`lsp-blue --TRANSITS(down)--> p-core-2` line. That single edge, with a property on it, is the
3 AM event — a lost label deep in the core, sitting in your graph as a fact you can now query
against. Before we query it, let's *see* it.


## See it — draw the graph

**Theory.** Text is great during an incident, but to *understand* a design you want the picture.
We'll colour nodes by their **label** (PEs, Ps, the LSP, the VRF, the Service each get a colour)
and draw the one **down** transit as a dashed red arrow so the failure jumps out. This is the
same instinct as a tunnel diagram — except this diagram is generated *from the data*, so it can
never drift out of sync with the truth.


In [ ]:
def draw(G, title='MPLS L3VPN knowledge graph'):
    colors = {'PE': '#0f7f74', 'P': '#3aa0ff', 'LSP': '#7d5bd0',
              'VRF': '#9aa5ad', 'Service': '#c8702a'}
    node_colors = [colors.get(G.nodes[n].get('label'), '#cccccc') for n in G]
    pos = nx.spring_layout(G, seed=7)   # fixed seed => stable, repeatable layout

    plt.figure(figsize=(10, 7))
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1900, alpha=0.92)
    nx.draw_networkx_labels(G, pos, font_size=9)

    down  = [(u, v) for u, v, d in G.edges(data=True) if d.get('state') == 'down']
    other = [(u, v) for u, v, d in G.edges(data=True) if (u, v) not in down]
    nx.draw_networkx_edges(G, pos, edgelist=other, arrows=True, arrowsize=16, edge_color='#8894a0')
    nx.draw_networkx_edges(G, pos, edgelist=down,  arrows=True, arrowsize=16,
                           edge_color='#d34b3f', width=2.0, style='dashed')
    nx.draw_networkx_edge_labels(G, pos, font_size=7,
        edge_labels={(u, v): d.get('rel', '') for u, v, d in G.edges(data=True)})

    plt.axis('off'); plt.title(title); plt.tight_layout(); plt.show()

draw(G)

**Reading the picture.** You should see the two PEs (teal) and two P routers (blue), the purple
`lsp-blue` tunnel, the grey `vrf-acme`, and the orange `Acme Payments` service — with one
**dashed red** arrow from `lsp-blue` to `p-core-2`, the broken hop. Trace it with your finger:
`Acme Payments -> vrf-acme -> lsp-blue -> ...p-core-2 (down)`. The picture already *implies* the
answer. In the next step we make the computer state it.


## Step 5 — The 3 AM question: blast radius as a traversal

**Theory.** Everything so far was setup for this. A **traversal** is just walking edges to answer
a question — and *you* decide the walk. Ours answers:

> *"If a transport LSP loses a hop, which VPN customers lose their path?"*

The route the walk takes, starting from the tunnel:

1. is this **LSP** broken? It is broken if its own `state='down'` **OR** any hop it `TRANSITS`
   is `down`. (One dead hop is enough — a tunnel is only as good as its worst hop.)
2. find the **VRFs** that ride it (`USES_LSP` edges pointing at the LSP);
3. note which **PE** each VRF sits `ON`;
4. and the **Services** that `RUNS_IN` those VRFs — the customers at risk.

Every hop is one edge. And the punchline is baked into step 1: the failing node is a **P router
with no customer config**, yet the walk still arrives at the customer's name — because the graph
carries the dependency the core router never could. Run it.


In [ ]:
def blast_radius(G):
    """Services put at risk by any broken transport LSP.

    An LSP is broken if its own state is 'down', OR if any hop it TRANSITS is 'down'.
    Returns (lsp, failed_hop, pe, vrf, service) tuples.
    """
    at_risk = []
    for lsp in G:
        if G.nodes[lsp].get('label') != 'LSP':
            continue
        # 1) is this LSP broken? collect any down hops
        down_hops = [v for _, v, d in G.out_edges(lsp, data=True)
                     if d.get('rel') == 'TRANSITS' and d.get('state') == 'down']
        lsp_down = G.nodes[lsp].get('state') == 'down' or bool(down_hops)
        if not lsp_down:
            continue
        failed_hop = down_hops[0] if down_hops else '(lsp state down)'
        # 2) VRFs riding this LSP
        for vrf, _, d2 in G.in_edges(lsp, data=True):
            if d2.get('rel') != 'USES_LSP':
                continue
            # 3) which PE the VRF sits on
            pe = next((p for _, p, d in G.out_edges(vrf, data=True) if d.get('rel') == 'ON'), '?')
            # 4) services running in that VRF
            for svc, _, d3 in G.in_edges(vrf, data=True):
                if d3.get('rel') == 'RUNS_IN':
                    at_risk.append((lsp, failed_hop, pe, vrf, svc))
    return at_risk

hits = blast_radius(G)
for lsp, hop, pe, vrf, svc in hits:
    print(f'{lsp} broken at {hop}  ->  {pe} / {vrf}  ->  AT RISK: {svc}')
if not hits:
    print('nothing at risk')

The forwarding table on `p-core-2` only ever showed you a missing label — one line, with no
customer anywhere in it. Your graph just told you **Acme Payments** is at risk, and showed its
work: the whole path from the dead hop back to the revenue. You got that answer because you gave
the graph the one node MPLS makes essential — the **LSP** — and wired the customer that rides it.
**That** is the underlay-dependency lesson: impact does not live where the config lives.


## Step 6 — What-if: repair the hop, then break it again

**Theory.** Because the failure lives on an edge as a plain property, you can *change* it and
re-ask — running "what if this core hop fails?" (or "what if I fix it?") on a **model**, safely,
with no one paged and no maintenance window. Flip the down transit to `up`: the LSP is whole
again and the blast radius clears. Flip it back to `down`: Acme Payments returns. This is the
humble beginning of pre-incident planning — you can ask the graph what *would* break before it
does.


In [ ]:
# You can read and write an edge's properties directly by [source][target].
G['lsp-blue']['p-core-2']['state'] = 'up'      # repair the hop
print('After repair:   ', blast_radius(G) or 'nothing at risk')

G['lsp-blue']['p-core-2']['state'] = 'down'    # break it again
print('After re-break: ', [svc for *_, svc in blast_radius(G)])

**Checkpoint.** After the repair you should see `nothing at risk`; after re-breaking it,
`['Acme Payments']` comes back. Same graph, same query — only the state on one edge changed. The
answer *responded*. That is what makes it a model rather than a drawing.


## Your turn #1 — a second customer on the same LSP

A transport LSP almost never carries just one customer — that's the whole economic point of
MPLS: many VPNs share the same core paths. Suppose **Beta Logistics**, on its own `vrf-beta`
sitting on `pe-east`, also rides `lsp-blue`. Add it, wire it up, and re-run `blast_radius`. You
should now see **two** customers surface from the *same* dead hop — because shared transport
means shared fate, and the graph knows it.

Fill in the cell below (there's a `# TODO`), then run it. The solution follows if you want to
check.


In [ ]:
# TODO: add a VRF 'vrf-beta' (label='VRF') and a Service 'Beta Logistics'
#       (label='Service', criticality='high'), then wire:
#         Beta Logistics -RUNS_IN-> vrf-beta
#         vrf-beta       -ON->      pe-east
#         vrf-beta       -USES_LSP-> lsp-blue   (rides the SAME tunnel)

# G.add_node(...); G.add_node(...)
# G.add_edge(...); G.add_edge(...); G.add_edge(...)

for lsp, hop, pe, vrf, svc in blast_radius(G):
    print(f'AT RISK: {svc}  (via {vrf} on {pe}, riding {lsp})')

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node('vrf-beta',       label='VRF')
G.add_node('Beta Logistics', label='Service', criticality='high')

G.add_edge('Beta Logistics', 'vrf-beta', rel='RUNS_IN')
G.add_edge('vrf-beta',       'pe-east',  rel='ON')
G.add_edge('vrf-beta',       'lsp-blue', rel='USES_LSP')

for lsp, hop, pe, vrf, svc in blast_radius(G):
    print(f'AT RISK: {svc}  (via {vrf} on {pe}, riding {lsp})')
```

Now both `Acme Payments` **and** `Beta Logistics` come back from the one dead hop on
`p-core-2`. The blast radius grew the instant you told the graph one more true thing — you
didn't touch the query at all. That is the shared-fate risk of shared transport, made visible.
</details>


In [ ]:
# (Solution, applied — so the rest of the notebook has both customers in the graph.)
G.add_node('vrf-beta',       label='VRF')
G.add_node('Beta Logistics', label='Service', criticality='high')
G.add_edge('Beta Logistics', 'vrf-beta', rel='RUNS_IN')
G.add_edge('vrf-beta',       'pe-east',  rel='ON')
G.add_edge('vrf-beta',       'lsp-blue', rel='USES_LSP')
print('Customers at risk now:', sorted({svc for *_, svc in blast_radius(G)}))

## Quiet reveal — you have been writing an *ontology*

Time to name the thing you've been doing. Every step, you made two very different kinds of
decision, and it's worth seeing the seam between them:

- *"A `Service` `RUNS_IN` a `VRF`. A `VRF` sits `ON` a `PE` and `USES_LSP` an `LSP`. An `LSP`
  `TRANSITS` a `P`."* — these are the **rules**: which node types exist, which edges are
  allowed, what shape a valid fact takes. That is the **schema**. Its fancy name is an
  **ontology** — and here's the friendly definition: *an ontology is the RFC of your graph, the
  agreed vocabulary written down as structure.* You already trust RFC 4364 to say what a valid
  L3VPN looks like; an ontology does the same job for your graph.

- *"This particular tunnel is `lsp-blue`, its hop across `p-core-2` is `down`, and `vrf-acme`
  rides it."* — these are the **instances**: the actual data.

A rule of thumb that keeps the two straight forever: **if it has a hostname, an interface, or a
label value, it is data (an instance), never schema.** `LSP` is schema; `lsp-blue` is data.
`TRANSITS` is schema; "the blue LSP crosses p-core-2 and it's down" is data. Keep the vocabulary
small and stable; let the instances be many. That single discipline is the difference between a
graph you can grow for years and a swamp you abandon in a month.


## A peek at the real thing (1/2) — the same overlay in Neo4j + Cypher

We used networkx so you could run everything with zero setup. But the *shapes* you built are
exactly what you'd build in a real graph database like **Neo4j**, whose query language is
**Cypher**. Cypher is worth a glance because it reads almost like the arrows we've been drawing
— `(node)-[:EDGE]->(node)`.

Here is **Steps 2–4 (the overlay and its transport)** as Cypher. `MERGE` means "find this node
or create it if missing" (so re-running is safe); `SET` assigns properties. This is *reference
only* — you don't run it here, it just shows the same idea in the grown-up tool:

```cypher
MERGE (svc:Service {name: 'Acme Payments'})  SET svc.criticality = 'critical';
MERGE (vrf:VRF {id: 'vrf-acme'});
MERGE (pe:ProviderEdge {id: 'pe-east'});
MERGE (lsp:MPLSLSP {id: 'lsp-blue'})         SET lsp.state = 'up';
MERGE (p2:ProviderRouter {id: 'p-core-2'});

MERGE (svc)-[:RUNS_IN]->(vrf);
MERGE (vrf)-[:ON]->(pe);
MERGE (vrf)-[:USES_LSP]->(lsp);
// the failure, as a property on the relationship — same as our edge
MERGE (lsp)-[:TRANSITS {state: 'down'}]->(p2);
```

See it? `(lsp)-[:TRANSITS {state:'down'}]->(p2)` is character-for-character the same statement
as our `G.add_edge('lsp-blue', 'p-core-2', rel='TRANSITS', state='down')`. Same node, same edge,
same fact-on-the-edge — one just happens to live in a database that scales to millions of nodes.


## A peek at the real thing (2/2) — the 3 AM question in Cypher

Our `blast_radius` walk is a 20-line Python function. In Cypher, that same traversal is a few
lines, because in a graph database you **draw the shape you're looking for** and let the engine
find every match — no manual loops:

```cypher
MATCH (lsp:MPLSLSP)-[t:TRANSITS {state:'down'}]->(p:ProviderRouter)
MATCH (vrf:VRF)-[:USES_LSP]->(lsp)
MATCH (vrf)-[:ON]->(pe:ProviderEdge)
MATCH (svc:Service)-[:RUNS_IN]->(vrf)
RETURN p.id        AS failed_core_router,
       collect(DISTINCT svc.name) AS customers_at_risk;
```

(This peek covers the hop-failure case, to keep the pattern readable; a full port would also
match the LSP being down as a whole — `... OR lsp.state = 'down'` — exactly like the two-part
"broken" test in our Python `blast_radius`.)

Read the first line as a sentence: *"match an LSP whose TRANSITS edge to a core router is down."*
That's the same step 1 you wrote in Python — the pattern you `MATCH` **is** the traversal. Run it
against a real L3VPN dataset and `p-core-2` comes back with `Acme Payments` and `Beta Logistics`
among the customers at risk — the failing core router named, beside the customers it never knew
it carried. Different engine; identical thinking. If you can read the Python above, you can read
the Cypher — you already speak this language.


## Your turn #2 — make the query respond to a *different* failure

So far there's been one tunnel. Real cores carry many, and a healthy one should stay silent.
Let's prove the model reacts to reality, not to whatever we hard-coded:

1. add a **second, healthy tunnel** `lsp-green` (`state='up'`) that transits `p-core-1` (up) and a
   new core router `p-core-3` (up), carrying a new customer **Zephyr Retail** on `vrf-zephyr`
   (on `pe-west`, riding `lsp-green`);
2. re-run `blast_radius` — Zephyr Retail should **not** appear (its tunnel is whole);
3. now break one hop: set `lsp-green -> p-core-3` to `down`, and re-run — Zephyr Retail **appears**.

This is the whole point of Step 6, in your own hands: the answer follows the state, tunnel by
tunnel. Fill in the `# TODO`s, then run.


In [ ]:
# TODO 1: build a healthy second tunnel and its customer:
#   nodes:  'lsp-green' (label='LSP', state='up'), 'p-core-3' (label='P'),
#           'vrf-zephyr' (label='VRF'), 'Zephyr Retail' (label='Service', criticality='medium')
#   edges:  Zephyr Retail -RUNS_IN-> vrf-zephyr
#           vrf-zephyr    -ON->       pe-west
#           vrf-zephyr    -USES_LSP-> lsp-green
#           pe-west       -HAS_LSP->  lsp-green
#           lsp-green     -TRANSITS-> p-core-1   (state='up')
#           lsp-green     -TRANSITS-> p-core-3   (state='up')

# ... your G.add_node / G.add_edge calls here ...

print('With lsp-green healthy: ', sorted({s for *_, s in blast_radius(G)}))

# TODO 2: break a hop on the green tunnel and re-print:
#   G['lsp-green']['p-core-3']['state'] = 'down'
#   print('After breaking lsp-green:', sorted({s for *_, s in blast_radius(G)}))

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node('lsp-green',     label='LSP', state='up')
G.add_node('p-core-3',      label='P')
G.add_node('vrf-zephyr',    label='VRF')
G.add_node('Zephyr Retail', label='Service', criticality='medium')

G.add_edge('Zephyr Retail', 'vrf-zephyr', rel='RUNS_IN')
G.add_edge('vrf-zephyr',    'pe-west',    rel='ON')
G.add_edge('vrf-zephyr',    'lsp-green',  rel='USES_LSP')
G.add_edge('pe-west',       'lsp-green',  rel='HAS_LSP')
G.add_edge('lsp-green',     'p-core-1',   rel='TRANSITS', state='up')
G.add_edge('lsp-green',     'p-core-3',   rel='TRANSITS', state='up')

print('With lsp-green healthy: ', sorted({s for *_, s in blast_radius(G)}))
# -> Acme Payments, Beta Logistics  (Zephyr is safe: its tunnel is whole)

G['lsp-green']['p-core-3']['state'] = 'down'
print('After breaking lsp-green:', sorted({s for *_, s in blast_radius(G)}))
# -> now Zephyr Retail joins the list
```

Before you broke the hop, Zephyr Retail was *in the graph* but *not at risk* — the query
correctly ignored a healthy tunnel. The moment `lsp-green` lost a hop, the exact same query
surfaced it. You didn't teach the query about the green tunnel; you told the graph the truth,
and the traversal did the rest.
</details>


In [ ]:
# (Solution, applied — leaves the green tunnel healthy again so later cells start clean.)
G.add_node('lsp-green',     label='LSP', state='up')
G.add_node('p-core-3',      label='P')
G.add_node('vrf-zephyr',    label='VRF')
G.add_node('Zephyr Retail', label='Service', criticality='medium')
G.add_edge('Zephyr Retail', 'vrf-zephyr', rel='RUNS_IN')
G.add_edge('vrf-zephyr',    'pe-west',    rel='ON')
G.add_edge('vrf-zephyr',    'lsp-green',  rel='USES_LSP')
G.add_edge('pe-west',       'lsp-green',  rel='HAS_LSP')
G.add_edge('lsp-green',     'p-core-1',   rel='TRANSITS', state='up')
G.add_edge('lsp-green',     'p-core-3',   rel='TRANSITS', state='up')

G['lsp-green']['p-core-3']['state'] = 'down'
print('Zephyr failure surfaces:', sorted({s for *_, s in blast_radius(G)}))
G['lsp-green']['p-core-3']['state'] = 'up'   # restore health
print('Restored, at risk again:', sorted({s for *_, s in blast_radius(G)}))

## Make it yours — swap in *your* network

Now the moment it becomes your tool, not mine. Change the five values below to **one** VRF and
**one** service *you* actually run, riding **one** transport LSP across **one** core router. We
add the down transit hop for you, so your service shows up. Run it, and watch **your own service
name** come back from `blast_radius` — proof that the machine you built understands your network,
not just the demo's.

Keep it to the smallest slice that answers one real question. You can always add a backup LSP or
a second VRF tomorrow — that's the whole promise of a graph you can grow.


In [ ]:
# --- change these five values to your VPN ---
MY_PE      = 'pe-lon-1'
MY_P       = 'p-fra-2'
MY_LSP     = 'lsp-gold'
MY_VRF     = 'vrf-globex'
MY_SERVICE = 'Globex Ledger'
# --------------------------------------------

G.add_node(MY_PE,      label='PE')
G.add_node(MY_P,       label='P')
G.add_node(MY_LSP,     label='LSP', state='up')
G.add_node(MY_VRF,     label='VRF')
G.add_node(MY_SERVICE, label='Service', criticality='critical')

G.add_edge(MY_SERVICE, MY_VRF, rel='RUNS_IN')
G.add_edge(MY_VRF, MY_PE,  rel='ON')
G.add_edge(MY_VRF, MY_LSP, rel='USES_LSP')
G.add_edge(MY_PE,  MY_LSP, rel='HAS_LSP')
G.add_edge(MY_LSP, MY_P,   rel='TRANSITS', state='down')   # your modelled failure

print(f'If {MY_P} loses labels for {MY_LSP}, these customers are at risk:')
for lsp, hop, pe, vrf, svc in blast_radius(G):
    if lsp == MY_LSP:
        print(f'  AT RISK: {svc}   (via {vrf} on {pe})')

**Checkpoint.** Your own service name — `Globex Ledger`, or whatever you typed — prints as *at
risk*, traced back to a core router (`p-fra-2`, or yours) that holds none of its config. That is
the moment the tool stopped being a tutorial and became yours. Every other customer you run is
just five more lines away.


## The one rule that keeps this from becoming a swamp

You'll be tempted to model everything MPLS touches — every label, every LFIB row, every RSVP
message. Don't. Here is the discipline, in one line:

> **Model the nouns of your design review, not the numbers of your forwarding plane.**

PEs, P routers, LSPs, VRFs, services, route targets, policies — the **nouns** you'd draw on a
whiteboard while arguing about a design — those belong in the graph. Per-label counters, LFIB
dumps, RSVP-TE state churn, interface utilisation — those are the **numbers**; leave them in the
systems that already store them well, and let the graph *reference* them when it needs to. The
graph holds the *shape of the dependency*; your NMS and the routers hold the firehose. Keep that
line sharp and your graph stays queryable for years.

### Where to go next
- **Add the underlay you already built.** That `lsp-blue` tunnel is signalled *over the IGP* —
  the **OSPF** graph from the companion lab. Chain them and one graph answers a question that
  spans both layers: an **OSPF area going dark** breaks the **LSP** that a **VPN** rides that a
  customer's **payments** depend on. Same five words, two layers.
- **Add protection.** Model a backup LSP as a second path the VRF *could* use, and "is this
  customer single-homed onto one tunnel?" becomes one more traversal.
- **Add the change layer.** Link a `ChangeEvent` node to the LSP it touched, and "what changed
  right before this tunnel broke?" becomes a query instead of a war-room guess.
- **The companion Neo4j lab** does this exact model in **Cypher** against a real graph database —
  same nodes, same edges, same blast-radius question — so the two Cypher peeks become things you
  run.


## You built an MPLS knowledge graph

Look back at the distance. You started with an empty page and a handful of plain words. You
added a provider fabric of PEs and Ps, a customer's VRF and service at the edge, and the tunnel
that carries them across the core — then you hung the failure on a single `TRANSITS` edge deep
in that core. Finally you asked the question no forwarding table can answer, and it printed
**Acme Payments** — then responded correctly every time you changed the world underneath it.

The important idea was never MPLS, and never networkx. It's this: **operational truth has a
shape** — a service runs in a VRF, a VRF rides an LSP, an LSP transits the core — and once that
shape is a graph, impact analysis stops being tribal knowledge and becomes a traversal. The
sharpest lesson MPLS taught us here is the underlay one: **the box that fails and the customer
that suffers can be miles apart in the config, but one edge apart in the graph.**

Your routers swap labels all day and never once know whose payments ride them. Now you know how
to build the graph that remembers. Go model one real customer on your network, and ask it what
breaks.
